In [ ]:
!pip uninstall -y accelerate peft transformers -q
!pip install -q \
    transformers==4.36.2 \
    accelerate==0.25.0 \
    peft==0.7.1 \
    datasets==2.16.1 \
    torchvision \
    pillow \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/SpotFakePlus_MirageNews"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

Mounted at /content/drive
Output dir: /content/drive/MyDrive/SpotFakePlus_MirageNews


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from PIL import Image

from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Device: cuda


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

In [ ]:
def preprocess(example):
    # ---- TEXT ----
    text_inputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # ---- IMAGE ----
    image = example["image"]
    if image is None or not isinstance(image, Image.Image):
        image = Image.new("RGB", (224, 224), (0, 0, 0))
    else:
        image = image.convert("RGB")

    image_inputs = image_processor(image, return_tensors="pt")

    return {
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
        "pixel_values": image_inputs["pixel_values"][0],
        "labels": example["label"]
    }

encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
from torchvision import models

class SpotFakePlus(nn.Module):
    def __init__(self):
        super().__init__()

        # TEXT ENCODER
        self.text_encoder = AutoModel.from_pretrained("bert-base-uncased")
        text_dim = self.text_encoder.config.hidden_size  # 768

        # IMAGE ENCODER
        self.image_encoder = models.resnet18(pretrained=True)
        image_dim = self.image_encoder.fc.in_features   # 512
        self.image_encoder.fc = nn.Identity()

        # FUSION CLASSIFIER
        self.classifier = nn.Linear(text_dim + image_dim, 2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        text_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_out.last_hidden_state[:, 0, :]  # CLS

        image_feat = self.image_encoder(pixel_values)

        fused = torch.cat([text_feat, image_feat], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = SpotFakePlus().to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 142MB/s]


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": roc_auc_score(labels, probs[:, 1])
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
image_processor.save_pretrained(OUTPUT_DIR)

print("✅ Model đã được lưu tại:", OUTPUT_DIR)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.211000,0.056081,0.979600,0.981526,0.977600,0.979559,0.998095
2,0.062100,0.066807,0.976400,0.972244,0.980800,0.976503,0.997641
3,0.032800,0.068347,0.977600,0.978365,0.976800,0.977582,0.997952


{'eval_loss': 0.05608099326491356, 'eval_accuracy': 0.9796, 'eval_precision': 0.9815261044176706, 'eval_recall': 0.9776, 'eval_f1': 0.9795591182364729, 'eval_auc': 0.99809504, 'eval_runtime': 257.221, 'eval_samples_per_second': 9.719, 'eval_steps_per_second': 0.307, 'epoch': 3.0}
✅ Model đã được lưu tại: /content/drive/MyDrive/SpotFakePlus_MirageNews


In [ ]:
# In params
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")

print("✅ Model đã được lưu tại:", OUTPUT_DIR)

Total params    : 120.66M
Trainable params: 120.66M
✅ Model đã được lưu tại: /content/drive/MyDrive/SpotFakePlus_MirageNews


In [ ]:
# ─── 2. Setup ─────────────────────────────────────────────────────────────────
import os, time
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import AutoTokenizer, AutoImageProcessor, AutoModel
from datasets import load_dataset

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/SpotFakePlus_MirageNews"
device   = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 3. Model class ───────────────────────────────────────────────────────────
class SpotFakePlus(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder     = AutoModel.from_pretrained("bert-base-uncased")
        text_dim              = self.text_encoder.config.hidden_size       # 768
        self.image_encoder    = models.resnet18(pretrained=False)
        image_dim             = self.image_encoder.fc.in_features          # 512
        self.image_encoder.fc = nn.Identity()
        self.classifier       = nn.Linear(text_dim + image_dim, 2)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        text_feat  = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]
        image_feat = self.image_encoder(pixel_values.float())  # ← fix Double→Float
        logits     = self.classifier(torch.cat([text_feat, image_feat], dim=1))
        loss       = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

# ─── 4. Load model từ Drive ───────────────────────────────────────────────────
print("⏳ Loading Spot-fake+...")
tokenizer       = AutoTokenizer.from_pretrained(SAVE_DIR)
image_processor = AutoImageProcessor.from_pretrained(SAVE_DIR)

model = SpotFakePlus()
state = load_file(os.path.join(SAVE_DIR, "model.safetensors"))
model.load_state_dict(state, strict=False)
model = model.to(device)
model.eval()
print("✅ Model loaded from Drive")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# ─── 5. Preprocess + Evaluate helpers ────────────────────────────────────────
def make_test_dataset(split_name):
    ds = load_dataset("anson-huang/mirage-news", split=split_name)
    def preprocess(example):
        text_inputs = tokenizer(
            example["text"],
            truncation=True, padding="max_length", max_length=128
        )
        image = example["image"]
        if image is None or not isinstance(image, Image.Image):
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        else:
            image = image.convert("RGB")
        image_inputs = image_processor(image, return_tensors="pt")
        return {
            "input_ids":      text_inputs["input_ids"],
            "attention_mask": text_inputs["attention_mask"],
            "pixel_values":   image_inputs["pixel_values"][0],
            "labels":         example["label"]
        }
    encoded = ds.map(preprocess, remove_columns=ds.column_names, desc=split_name)
    return encoded

def collate_fn(batch):
    result = {}
    for k in batch[0]:
        arr = np.array([b[k] for b in batch])
        t   = torch.tensor(arr)
        if k == "pixel_values":
            t = t.float()   # ← fix Double→Float
        result[k] = t
    return result

def evaluate_split(dataset):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                pixel_values   = batch["pixel_values"].to(device)
            )
            all_logits.append(out["logits"].cpu())
            all_labels.append(batch["labels"])
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 6. Test Evaluation ───────────────────────────────────────────────────────
test_splits = {
    "ID  test1_nyt_mj":    "test1_nyt_mj",
    "OOD test2_bbc_dalle": "test2_bbc_dalle",
    "OOD test3_cnn_dalle": "test3_cnn_dalle",
    "OOD test4_bbc_sdxl":  "test4_bbc_sdxl",
    "OOD test5_cnn_sdxl":  "test5_cnn_sdxl",
}

print("\n📊 Test Evaluation — Spot-fake+ (MIRAGE-News):")
print(f"{'Split':<26} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>8}")
print("-" * 63)

results   = {}
ood_names = []

for name, split in test_splits.items():
    td            = make_test_dataset(split)
    m             = evaluate_split(td)
    results[name] = m
    print(f"{name:<26} {m['accuracy']*100:>6.2f}% {m['precision']*100:>6.2f}% "
          f"{m['recall']*100:>6.2f}% {m['f1']*100:>6.2f}% {m['auc']:>8.5f}")
    if name.startswith("OOD"):
        ood_names.append(name)

print("-" * 63)
ood_avg = {m: np.mean([results[n][m] for n in ood_names])
           for m in ["accuracy", "precision", "recall", "f1", "auc"]}
print(f"{'OOD Average':<26} {ood_avg['accuracy']*100:>6.2f}% {ood_avg['precision']*100:>6.2f}% "
      f"{ood_avg['recall']*100:>6.2f}% {ood_avg['f1']*100:>6.2f}% {ood_avg['auc']:>8.5f}")

# ─── 7. Latency + VRAM ───────────────────────────────────────────────────────
dummy_text  = "This is a sample news headline for inference measurement."
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))

text_inp  = tokenizer(dummy_text, truncation=True, padding="max_length",
                      max_length=128, return_tensors="pt")
image_inp = image_processor(dummy_image, return_tensors="pt")

input_ids      = text_inp["input_ids"].to(device)
attention_mask = text_inp["attention_mask"].to(device)
pixel_values   = image_inp["pixel_values"].float().to(device)  # ← float()

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids, attention_mask, pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids, attention_mask, pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0
print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
⏳ Loading Spot-fake+...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_no

✅ Model loaded from Drive
Total params    : 120,661,314
Trainable params: 120,661,314

📊 Test Evaluation — Spot-fake+ (MIRAGE-News):
Split                          Acc    Prec     Rec      F1      AUC
---------------------------------------------------------------


...

OOD test3_cnn_dalle         88.00%  87.70%  88.40%  88.05%  0.94390


test4_bbc_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test4_bbc_sdxl          82.00%  74.24%  98.00%  84.48%  0.94909


test5_cnn_sdxl:   0%|          | 0/500 [00:00<?, ? examples/s]

OOD test5_cnn_sdxl          89.60%  86.67%  93.60%  90.00%  0.96522
---------------------------------------------------------------
OOD Average                 85.05%  80.76%  93.30%  86.33%  0.94428

Total params    : 120,661,314
Trainable params: 120,661,314
Latency median  : 412.06 ms
VRAM (peak)     : 0.0 MB


In [ ]:
import os, torch, time, numpy as np
from torch import nn
from transformers import AutoTokenizer, AutoImageProcessor, AutoModel
from torchvision import models
from safetensors.torch import load_file
from PIL import Image

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

N_WARMUP = 10
N_RUNS   = 100

ckpt_dir = "/content/drive/MyDrive/SpotFakePlus_MirageNews"

# ─── Model class ─────────────────────────────────────────────────────────────
class SpotFakePlus(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder     = AutoModel.from_pretrained("bert-base-uncased")
        text_dim              = self.text_encoder.config.hidden_size
        self.image_encoder    = models.resnet18(pretrained=False)
        image_dim             = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()
        self.classifier       = nn.Linear(text_dim + image_dim, 2)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        text_feat  = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        image_feat = self.image_encoder(pixel_values)
        logits     = self.classifier(torch.cat([text_feat, image_feat], dim=1))
        loss       = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

# ─── Load model ───────────────────────────────────────────────────────────────
print("⏳ Loading Spot-fake+...")
model = SpotFakePlus()
state = load_file(f"{ckpt_dir}/model.safetensors")
model.load_state_dict(state, strict=False)
model = model.to(device)
model.eval()

# In params
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")

tokenizer       = AutoTokenizer.from_pretrained("bert-base-uncased")
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
print("✅ Model loaded")

# ─── Dummy input ─────────────────────────────────────────────────────────────
dummy_image  = Image.new("RGB", (224, 224), color=(128, 128, 128))
dummy_text   = "This is a sample news headline for inference measurement."

text_inputs  = tokenizer(dummy_text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
image_inputs = image_processor(dummy_image, return_tensors="pt")

input_ids      = text_inputs["input_ids"].to(device)
attention_mask = text_inputs["attention_mask"].to(device)
pixel_values   = image_inputs["pixel_values"].to(device)

# ─── Đo ──────────────────────────────────────────────────────────────────────
try:
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

    latencies = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            if device == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
            if device == "cuda": torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)

    latencies = np.array(latencies)
    vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

    print(f"\n{'='*50}")
    print(f"📊 Spot-fake+")
    print(f"{'='*50}")
    print(f"  Latency — mean: {latencies.mean():.2f} ms | median: {np.median(latencies):.2f} ms | std: {latencies.std():.2f} ms")
    print(f"  VRAM (peak allocated): {vram:.1f} MB")

except Exception as e:
    print(f"❌ Lỗi: {e}")
finally:
    model.cpu()
    if device == "cuda": torch.cuda.empty_cache()

Using device: cuda
⏳ Loading Spot-fake+...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Total params    : 120.66M
Trainable params: 120.66M


Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


✅ Model loaded

📊 Spot-fake+
  Latency — mean: 34.06 ms | median: 30.63 ms | std: 14.01 ms
  VRAM (peak allocated): 1918.1 MB
